# Path 1 - Journey-ID bug

In our previous analysis, there was a notice that the journey ID logic was implemented incorrectly.  We will set up a toy example including 8 journeys that cover 60 days worth of data and attempt to correct the issue.

The journeys will be as follows:
- A: view->purchase
- B: view-view->purchase
- C: view->purchase->view->cart->purchase
- D: view .... 35 days ... view
- E: view->cart (no purchase)
- F: purchase (no view)
- G: view ... 30 days ... view
- H: view ... 3 days ... view ... 29 days ... purchase

All journeys will start on 1/1/26 and be done by different user/product combinations

In [15]:
import numpy as np
import pandas as pd
import datetime as dt
import duckdb

In [12]:
users = np.random.randint(50, 100, 25)
products = np.random.randint(225,500, 30)

In [13]:
journey_user = np.random.choice(users, 8)
journey_product = np.random.choice(products, 8)

In [33]:
dummy = pd.DataFrame([
    # journey A
    {'event_time': dt.datetime(2026,1,1,8,0,0), 'event_type': 'view', 'user_id': journey_user[0], 'product_id': journey_product[0]},
    {'event_time': dt.datetime(2026,1,1,8,30,0), 'event_type': 'purchase', 'user_id': journey_user[0], 'product_id': journey_product[0]},

    #journey B
    {'event_time': dt.datetime(2026,1,1,9,0,0), 'event_type': 'view', 'user_id': journey_user[1], 'product_id': journey_product[1]},
    {'event_time': dt.datetime(2026,1,15,9,0,0), 'event_type': 'view', 'user_id': journey_user[1], 'product_id': journey_product[1]},
    {'event_time': dt.datetime(2026,1,20,9,0,0), 'event_type': 'purchase', 'user_id': journey_user[1], 'product_id': journey_product[1]},

    #journey C
    {'event_time': dt.datetime(2026,1,1,8,15,0), 'event_type': 'view', 'user_id': journey_user[2], 'product_id': journey_product[2]},
    {'event_time': dt.datetime(2026,1,1,8,30,0), 'event_type': 'purchase', 'user_id': journey_user[2], 'product_id': journey_product[2]},
    {'event_time': dt.datetime(2026,1,2,12,0,0), 'event_type': 'view', 'user_id': journey_user[2], 'product_id': journey_product[2]},
    {'event_time': dt.datetime(2026,1,5,8,0,0), 'event_type': 'cart', 'user_id': journey_user[2], 'product_id': journey_product[2]},
    {'event_time': dt.datetime(2026,1,10,8,0,0), 'event_type': 'purchase', 'user_id': journey_user[2], 'product_id': journey_product[2]},

    # journey D
    {'event_time': dt.datetime(2026,1,1,8,0,0), 'event_type': 'view', 'user_id': journey_user[3], 'product_id': journey_product[3]},
    {'event_time': dt.datetime(2026,2,6,8,0,0), 'event_type': 'view', 'user_id': journey_user[3], 'product_id': journey_product[3]},

    # journey E
    {'event_time': dt.datetime(2026,1,1,10,0,0), 'event_type': 'view', 'user_id': journey_user[4], 'product_id': journey_product[4]},
    {'event_time': dt.datetime(2026,1,1,20,0,0), 'event_type': 'cart', 'user_id': journey_user[4], 'product_id': journey_product[4]},

    #journey F
    {'event_time': dt.datetime(2026,1,1,8,0,0), 'event_type': 'purchase', 'user_id': journey_user[5], 'product_id': journey_product[5]},

    # journey G
    {'event_time': dt.datetime(2026,1,1,8,0,0), 'event_type': 'view', 'user_id': journey_user[6], 'product_id': journey_product[6]},
    {'event_time': dt.datetime(2026,2,1,8,0,0), 'event_type': 'view', 'user_id': journey_user[6], 'product_id': journey_product[6]},

    # journey H
    {'event_time': dt.datetime(2026,1,1,8,0,0), 'event_type': 'view', 'user_id': journey_user[7], 'product_id': journey_product[7]},
    {'event_time': dt.datetime(2026,1,4,8,0,0), 'event_type': 'view', 'user_id': journey_user[7], 'product_id': journey_product[7]},
    {'event_time': dt.datetime(2026,2,2,8,0,0), 'event_type': 'purchase', 'user_id': journey_user[7], 'product_id': journey_product[7]}
])


In [34]:
duckdb.sql('create table journey_test as select * from dummy')

CatalogException: Catalog Error: Table with name "journey_test" already exists!

In [159]:
# now that we have our dummy table, let's examine the previous methodology for generating user journeys
duckdb.sql("""
CREATE OR REPLACE TABLE journeys AS SELECT *,
SUM(case when event_type = 'purchase' then 1 else 0 end)
OVER (PARTITION BY user_id, product_id ORDER BY event_time 
RANGE BETWEEN CURRENT ROW
              AND INTERVAL '30 days' FOLLOWING) AS journey_num
from journey_test
""")

In [160]:
#duckdb.sql("UPDATE journeys SET journey_num = journey_num - 1 WHERE event_type='purchase'")
duckdb.sql("ALTER TABLE journeys ADD COLUMN IF NOT EXISTS converted BOOL DEFAULT false")
duckdb.sql("ALTER TABLE journeys ADD COLUMN IF NOT EXISTS journey_id STRING")
duckdb.sql("UPDATE journeys SET journey_id = CONCAT(user_id,'-',product_id,'-',journey_num)")
duckdb.sql("UPDATE journeys SET converted = TRUE WHERE journey_id IN (select journey_id FROM journeys WHERE event_type='purchase')")

In [161]:
duckdb.sql('select * from journeys order by journey_id, event_time').df()

,event_time,event_type,user_id,product_id,journey_num,converted,journey_id
0,2026-01-01 10:00:00,view,54,326,0.0,False,54-326-0
1,2026-01-01 20:00:00,cart,54,326,0.0,False,54-326-0
2,2026-01-01 08:00:00,purchase,65,337,1.0,True,65-337-1
3,2026-01-02 12:00:00,view,66,374,1.0,True,66-374-1
4,2026-01-05 08:00:00,cart,66,374,1.0,True,66-374-1
5,2026-01-10 08:00:00,purchase,66,374,1.0,True,66-374-1
6,2026-01-01 08:15:00,view,66,374,2.0,True,66-374-2
7,2026-01-01 08:30:00,purchase,66,374,2.0,True,66-374-2
8,2026-01-01 08:00:00,view,74,350,1.0,True,74-350-1
9,2026-01-01 08:30:00,purchase,74,350,1.0,True,74-350-1


Definitely, removing the subtracting 1 is getting things closer.  I was under the impression that it added 1 before writing the purchase, so that it would start a new journey number.  However, this general tack does not seem to be working.  Let's try starting with removing the purchases and see how that works to start, then maybe try assigning journey numbers based on 30 days past the minimum date.  It still seems like we would need to run a while loop in order to get them all, since we'll be continually removing a journey and starting a new one every 30 days.

In [162]:
purchases = duckdb.sql("""
select *,
COALESCE(lag(event_time) over (partition by user_id, product_id order by event_time),
event_time - INTERVAL 30 DAYS) as journey_begin,
uuidv4() as journey_id
from (
select *,
sum(1) over (partition by user_id, product_id order by event_time) as journey_num,
from journey_test where event_type='purchase'
)
""").df()
purchases

,event_time,event_type,user_id,product_id,journey_num,journey_begin,journey_id
0,2026-02-02 08:00:00,purchase,78,309,1.0,2026-01-03 08:00:00,1712810a-4937-4876-832a-a70c5e0971ba
1,2026-01-01 08:30:00,purchase,66,374,1.0,2025-12-02 08:30:00,eec1c3c7-6fb6-46c7-87ac-fb3948170183
2,2026-01-10 08:00:00,purchase,66,374,2.0,2026-01-01 08:30:00,9ed8be0f-507e-4a37-85b1-1dec3a05f09c
3,2026-01-20 09:00:00,purchase,86,342,1.0,2025-12-21 09:00:00,3f6a92dd-3aae-4b81-9b3f-baa9c79c8e97
4,2026-01-01 08:00:00,purchase,65,337,1.0,2025-12-02 08:00:00,9d5ee35a-6a89-4205-b638-5fc81942ba01
5,2026-01-01 08:30:00,purchase,74,350,1.0,2025-12-02 08:30:00,33dd9618-187d-407d-8870-465cb25bec82


In [163]:
duckdb.sql("""create or replace table purchase_journeys as
select a.*, b.journey_id
from journey_test a left join purchases b on
a.user_id=b.user_id and a.product_id=b.product_id
and (a.event_time > b.journey_begin and a.event_time <= b.event_time)""")

In [164]:
duckdb.sql("select * from purchase_journeys").df()

,event_time,event_type,user_id,product_id,journey_id
0,2026-01-01 08:00:00,view,74,350,33dd9618-187d-407d-8870-465cb25bec82
1,2026-01-01 08:30:00,purchase,74,350,33dd9618-187d-407d-8870-465cb25bec82
2,2026-01-01 09:00:00,view,86,342,3f6a92dd-3aae-4b81-9b3f-baa9c79c8e97
3,2026-01-15 09:00:00,view,86,342,3f6a92dd-3aae-4b81-9b3f-baa9c79c8e97
4,2026-01-20 09:00:00,purchase,86,342,3f6a92dd-3aae-4b81-9b3f-baa9c79c8e97
5,2026-01-02 12:00:00,view,66,374,9ed8be0f-507e-4a37-85b1-1dec3a05f09c
6,2026-01-05 08:00:00,cart,66,374,9ed8be0f-507e-4a37-85b1-1dec3a05f09c
7,2026-01-10 08:00:00,purchase,66,374,9ed8be0f-507e-4a37-85b1-1dec3a05f09c
8,2026-01-01 08:00:00,purchase,65,337,9d5ee35a-6a89-4205-b638-5fc81942ba01
9,2026-01-04 08:00:00,view,78,309,1712810a-4937-4876-832a-a70c5e0971ba


now we need to handle the non-purchases.  We will continue to run a "select latest, count back 30 days, assign" until there are no uncategorized transactions

In [166]:
while duckdb.sql('select * from purchase_journeys where journey_id is null').fetchone() != None:
    # generate new journey ids
    tmp = duckdb.sql("""
select max(event_time) as event_time, 
max(event_time) - INTERVAL 30 DAYS as journey_begin,
user_id, product_id, 
uuidv4() as journey_id
from purchase_journeys 
where journey_id is null
group by user_id, product_id""").df()

    # update uncoverted
    duckdb.sql("""update purchase_journeys orig
set journey_id = d.journey_id from (
select a.event_time, a.event_type, a.user_id, a.product_id,
coalesce(a.journey_id, b.journey_id, NULL) as journey_id
from (select * from purchase_journeys where journey_id is null) a left join tmp b on
a.user_id=b.user_id and a.product_id=b.product_id
and (a.event_time > b.journey_begin and a.event_time <= b.event_time)) as d
where orig.user_id = d.user_id and orig.product_id = d.product_id
and orig.event_time = d.event_time""")

    # debug - number of rows remaining
    print("Rows remaining to assign:")
    print(duckdb.sql("select count(*) from purchase_journeys where journey_id is null"))

Rows remaining to assign:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            2 │
└──────────────┘

Rows remaining to assign:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘



In [167]:
duckdb.sql('select * from purchase_journeys order by journey_id, event_time').df()

,event_time,event_type,user_id,product_id,journey_id
0,2026-01-04 08:00:00,view,78,309,1712810a-4937-4876-832a-a70c5e0971ba
1,2026-02-02 08:00:00,purchase,78,309,1712810a-4937-4876-832a-a70c5e0971ba
2,2026-01-01 08:00:00,view,74,350,33dd9618-187d-407d-8870-465cb25bec82
3,2026-01-01 08:30:00,purchase,74,350,33dd9618-187d-407d-8870-465cb25bec82
4,2026-01-01 09:00:00,view,86,342,3f6a92dd-3aae-4b81-9b3f-baa9c79c8e97
5,2026-01-15 09:00:00,view,86,342,3f6a92dd-3aae-4b81-9b3f-baa9c79c8e97
6,2026-01-20 09:00:00,purchase,86,342,3f6a92dd-3aae-4b81-9b3f-baa9c79c8e97
7,2026-01-01 08:00:00,view,97,432,45d11eab-39a1-42a9-8442-f6c799c57bda
8,2026-01-01 08:00:00,view,83,463,47406603-056a-4840-a668-7c8c65e14f60
9,2026-01-01 08:00:00,view,78,309,5563c793-bee7-4402-993a-b9e1792e9272


# fixing recommendations - resetting to testable hypotheses

## Price changes
I did not intend to put across such a strong recommendation to keep prices steady in general.  My intention was to keep prices steady while we continued to explore the issue.  That being said, several statistical tests could be used to analyze this situation.  The first being that we take a closer look at user journeys and see
where prices went up during the journey and where they went down.  We could then generate a more reasonable test going forward to test the determined effect.

This could be done in the form of an A/B test where we grant certain users a 5% or 10% discount on the price and see if they are more likely to convert (which I would expect to be likely).  We would want to check conversion rate as our success metric, with overall profitability as a guardrail.  Depending on exactly how things were set up, we could assign users to test groups and within those groups assume independence of journeys.  Under these conditions, we would need to see not only the expected lift in conversion rate, but also not violate our guardrail metric.

## Reminder recommendation
I understand how this could be moving too far too fast.  The appropriate recommendation would be to do an experiment based on the length of time we generally (on average) tend to see between sessions on active conversions.  This would be another easy A/B test by user, where we send a simple reminder email about the items in their cart.  Let's assume that the average length of time between sessions for conversions is 5 days.  If a user/product has not converted and they have 5 days since their last session in the journey, we send them a reminder email.  Success metric would obviously be conversion rate.  Guardrail metric would be site traffic, since running email reminders might be considered nagging and annoying and drive users away from the site.  Again, this would be segmented into test/control by user and we should assume independence between the different journeys as a fast first pass.  We might want to start stratifying by user if things look odd, but that would create a significant amount of extra overhead and required samples in order to successfully implement.

# Product sense
The Instant Buy launch criterion was a little weak.  We based the entire launch decision on the success criteria and completely ignored our guardrails and a more appropriate north star metric.  Obviously, a more appropriate launch criteria would be that we have a positive result on our success metric (we have significant positive effect lift to our success metric compared to the inherent noise and are able to reject the null hypothesis in favor of the alternative hypothesis) and fail to reject any null hypotheses on our guardrails (so that we aren't doing damage at the same time).

# Failed assumptions:
There was an initial assumption that we could organize our funnel by session only.  This assumption was made before taking a closer look at the data - there was an assumption that a cart checkout would include multiple products, but the data was more organized toward a single item per checkout.  This is why our assumption had to be changed in order to use a user/product funnel instead of a user/session funnel.

Next time, I think I would need to take a closer look at one of the sessions with a purchase before making assumptions as to how the data is organized.